# Fase 1B — Preprocesamiento para Transformers (IroSvA)

Este notebook prepara las entradas utilizadas por BETO, DistilBETO y RoBERTuito.

Se generan tres representaciones del texto:

- `MESSAGE_TRANSFORMER_BASE` (BETO y DistilBETO);
- `MESSAGE_TRANSFORMER_SOCIAL` (BETO y DistilBETO);
- `MESSAGE_ROBERTUITO` (solo si `pysentimiento` está instalado; si no, queda vacía).

Se mantiene el control de calidad interno del train (duplicados exactos y conflictos de etiqueta), la extracción de las 9 variables lingüísticas y el diagnóstico de tokenización.

El test se procesa con las mismas reglas, pero **no se utiliza para eliminar registros del train**.

La prevención de leakage entre folds se realiza posteriormente, en los notebooks de cada Transformer, mediante grupos de textos coincidentes + `StratifiedGroupKFold`.

## 1. Librerías y configuración

Las rutas, las variantes dialectales y los checkpoints se definen una sola vez en esta celda. Los CSV se exportan en `../data/transformers/`.

`ROBERTUITO_ALTERNATIVO` (`robertuito-base-deacc`) solo se registra como referencia; en este notebook no se utiliza.

In [ ]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import emoji
from IPython.display import display

try:
    from transformers import AutoTokenizer
    TRANSFORMERS_DISPONIBLE = True
except ImportError:
    AutoTokenizer = None
    TRANSFORMERS_DISPONIBLE = False

try:
    from pysentimiento.preprocessing import preprocess_tweet
    PYSENTIMIENTO_DISPONIBLE = True
except ImportError:
    preprocess_tweet = None
    PYSENTIMIENTO_DISPONIBLE = False

DATA_DIR = Path("../data")
OUT_DIR = DATA_DIR / "transformers"
OUT_DIR.mkdir(parents=True, exist_ok=True)

VARIANTES = ["mx", "es", "cu"]

CHECKPOINTS = {
    "BETO": "dccuchile/bert-base-spanish-wwm-cased",
    "DistilBETO": "dccuchile/distilbert-base-spanish-uncased",
    "RoBERTuito": "pysentimiento/robertuito-base-uncased",
}
ROBERTUITO_ALTERNATIVO = "pysentimiento/robertuito-base-deacc"

print("Datos de entrada:", DATA_DIR)
print("Salida:", OUT_DIR)
print("Variantes:", VARIANTES)
print("transformers disponible:", TRANSFORMERS_DISPONIBLE)
print("pysentimiento disponible:", PYSENTIMIENTO_DISPONIBLE)
for modelo, checkpoint in CHECKPOINTS.items():
    print(f"{modelo:12} -> {checkpoint}")
print(f"{'RoBERTuito alt.':12} -> {ROBERTUITO_ALTERNATIVO}")

## 2. Carga y auditoría del train oficial

Se cargan las tres variantes dialectales por separado.

Antes de eliminar registros se verifica, comparando el `MESSAGE` original de forma exacta:

- cuántos textos `MESSAGE` están repetidos;
- si un mismo texto repetido tiene **la misma etiqueta**;
- si existe un **conflicto de anotación** (mismo `MESSAGE` con etiquetas distintas).

Después de la auditoría:

1. los textos con etiquetas contradictorias se retiran **completamente** del train y se guardan en `conflictos_etiqueta_train.csv`;
2. de los duplicados restantes (misma etiqueta) se conserva una sola copia (la primera).

El test no interviene en este control. El mismo criterio se aplica en los preprocesamientos de ML y de Transformers.

In [ ]:
def auditar_duplicados_train(df, variante):
    """Cuenta textos repetidos y conflictos de etiqueta dentro del train de una variante."""
    grupos = (
        df.groupby("MESSAGE", dropna=False)["IS_IRONIC"]
        .agg(["size", "nunique"])
        .reset_index()
    )
    repetidos = grupos[grupos["size"] > 1]
    conflictos = repetidos[repetidos["nunique"] > 1]
    mismas_etiquetas = repetidos[repetidos["nunique"] == 1]

    return {
        "variante": variante,
        "n_original": len(df),
        "n_textos_repetidos": len(repetidos),
        "n_duplicados_misma_etiqueta": len(mismas_etiquetas),
        "n_conflictos_etiqueta": len(conflictos),
        "mensajes_conflicto": set(conflictos["MESSAGE"].tolist()),
    }


def cargar_train():
    """Carga el train de cada variante, retira conflictos y deduplica textos exactos."""
    frames = []
    auditoria = []
    conflictos_detalle = []

    for variante in VARIANTES:
        df = pd.read_csv(DATA_DIR / f"irosva.{variante}.training.csv")
        df["VARIANTE"] = variante

        info = auditar_duplicados_train(df, variante)
        mensajes_conflicto = info["mensajes_conflicto"]

        # Guardar TODAS las filas implicadas en conflictos.
        df_conflictos = df[df["MESSAGE"].isin(mensajes_conflicto)].copy()
        if not df_conflictos.empty:
            df_conflictos["MOTIVO_EXCLUSION"] = "mismo_texto_etiquetas_contradictorias"
            conflictos_detalle.append(df_conflictos)
            print(f"\n{variante.upper()} — conflictos de etiqueta:")
            display(
                df_conflictos[["ID", "MESSAGE", "IS_IRONIC", "VARIANTE"]]
                .sort_values("MESSAGE")
            )

        # 1. Retirar completamente los textos con etiquetas contradictorias.
        df_sin_conflictos = df[~df["MESSAGE"].isin(mensajes_conflicto)].copy()

        # 2. En los duplicados restantes la etiqueta coincide: conservar una copia.
        df_limpio = (
            df_sin_conflictos
            .drop_duplicates(subset="MESSAGE", keep="first")
            .reset_index(drop=True)
        )
        n_duplicados_retirados = len(df_sin_conflictos) - len(df_limpio)

        auditoria.append({
            "variante": variante,
            "n_original": info["n_original"],
            "textos_repetidos": info["n_textos_repetidos"],
            "conflictos_texto": info["n_conflictos_etiqueta"],
            "filas_retiradas_conflicto": len(df_conflictos),
            "filas_duplicadas_misma_etiqueta_retiradas": n_duplicados_retirados,
            "n_tras_control_duplicados": len(df_limpio),
        })

        print(
            f"[TRAIN][{variante.upper()}] {info['n_original']} originales | "
            f"{len(df_conflictos)} retirados por conflicto | "
            f"{n_duplicados_retirados} duplicados de misma etiqueta retirados | "
            f"{len(df_limpio)} restantes"
        )
        frames.append(df_limpio)

    df_train = pd.concat(frames, ignore_index=True)
    df_auditoria = pd.DataFrame(auditoria)
    df_conflictos_train = (
        pd.concat(conflictos_detalle, ignore_index=True)
        if conflictos_detalle
        else pd.DataFrame()
    )
    return df_train, df_auditoria, df_conflictos_train


df_train, df_auditoria_train, df_conflictos_train = cargar_train()

print("\nAUDITORÍA TRAIN — DUPLICADOS Y CONFLICTOS")
display(df_auditoria_train)

print(f"\n[TRAIN] Total tras control de duplicados/conflictos: {len(df_train)} registros")
print(df_train["VARIANTE"].value_counts().sort_index())

print("\nDistribución de clases:")
print(df_train.groupby("VARIANTE")["IS_IRONIC"].value_counts())

## 3. Carga del test oficial

El test se carga para generar las mismas representaciones que el train (`BASE`, `SOCIAL` y `ROBERTUITO`).

No se utiliza para modificar la composición del train.

In [ ]:
def cargar_test(variante):
    """Une los textos del test con sus etiquetas oficiales."""
    df_texto = pd.read_csv(DATA_DIR / f"irosva.{variante} - irosva.{variante}.test.csv")
    df_truth = pd.read_csv(DATA_DIR / f"irosva.{variante}.test.truth.csv")

    if "IS_IRONIC" in df_texto.columns:
        df_texto = df_texto.drop(columns=["IS_IRONIC"])

    df = df_texto.merge(df_truth[["ID", "IS_IRONIC"]], on="ID", how="inner")
    assert len(df) == len(df_texto), f"{variante}: hay textos del test sin etiqueta oficial"

    df["VARIANTE"] = variante
    return df


df_test = pd.concat([cargar_test(v) for v in VARIANTES], ignore_index=True)

print(f"[TEST] Total: {len(df_test)} registros")
print(df_test["VARIANTE"].value_counts().sort_index())

## 4. Variables lingüísticas

Se extraen las mismas 9 variables lingüísticas del experimento ML, calculadas sobre el texto original, antes de cualquier limpieza.

**No se incorporan en el primer baseline Transformer.** Se guardan para un experimento posterior: Transformer puro vs Transformer + variables lingüísticas.

(González-Ibáñez et al., 2011; Joshi et al., 2015, 2016; Šandor & Bagić Babac, 2023; RAE, 2011)

In [ ]:
def quitar_tildes(texto):
    """Convierte caracteres acentuados a su forma base para capturar
    variantes de escritura informal (jamás→jamas, ningún→ningun)."""
    return "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )


# Negaciones en español.
# Fuente: RAE y Asociación de Academias de la Lengua Española (2011).
# Nueva gramática básica de la lengua española.
NEGACIONES = {
    "no", "nunca", "jamas", "tampoco",
    "nada", "nadie",
    "ningun", "ninguno", "ninguna", "ningunos", "ningunas",
    "ni", "sino", "sin",
}

FEATURE_COLS = [
    "n_exc", "n_int", "n_may", "n_emo", "n_ris",
    "n_neg", "n_elo", "n_com", "n_pun",
]


def extraer_features_linguisticos(texto):
    """Extrae 9 variables lingüísticas como conteos sobre el texto original."""
    texto_str = str(texto)
    texto_norm = quitar_tildes(texto_str.lower())
    palabras = re.findall(r"\b\w+\b", texto_norm)

    # 1. Exclamaciones (González-Ibáñez et al., 2011; Joshi et al., 2015)
    n_exc = texto_str.count("!") + texto_str.count("¡")
    # 2. Interrogaciones (González-Ibáñez et al., 2011; Joshi et al., 2015)
    n_int = texto_str.count("?") + texto_str.count("¿")
    # 3. Mayúsculas enfáticas: 3+ caracteres (Joshi et al., 2015; Šandor & Bagić Babac, 2023)
    n_may = len(re.findall(r"\b[A-ZÁÉÍÓÚÜÑ]{3,}\b", texto_str))
    # 4. Emojis (González-Ibáñez et al., 2011; Joshi et al., 2015)
    n_emo = sum(1 for c in texto_str if c in emoji.EMOJI_DATA)
    # 5. Risas (González-Ibáñez et al., 2011)
    n_ris = len(re.findall(r"\b(ja+ja+|je+je+|ji+ji+|ha+ha+|xd+|lol+|jsjs)\b", texto_norm))
    # 6. Negaciones (González-Ibáñez et al., 2011; RAE, 2011)
    n_neg = sum(1 for p in palabras if p in NEGACIONES)
    # 7. Elongación: mismo carácter 3+ veces (Šandor & Bagić Babac, 2023; Joshi et al., 2016)
    n_elo = len(re.findall(r"(.)\1{2,}", texto_norm))
    # 8. Comillas (González-Ibáñez et al., 2011; Šandor & Bagić Babac, 2023)
    n_com = len(re.findall(r'["\'\u2018\u2019\u201c\u201d«»]', texto_str))
    # 9. Puntos suspensivos (Šandor & Bagić Babac, 2023; Joshi et al., 2016)
    n_pun = texto_str.count("...") + texto_str.count("\u2026")

    return [n_exc, n_int, n_may, n_emo, n_ris, n_neg, n_elo, n_com, n_pun]

In [ ]:
for nombre, df in [("Train", df_train), ("Test", df_test)]:
    features = df["MESSAGE"].fillna("").apply(extraer_features_linguisticos).tolist()
    for i, col in enumerate(FEATURE_COLS):
        df[col] = [f[i] for f in features]

    print(f"\n[FEATURES][{nombre.upper()}] Medias:")
    print(df[FEATURE_COLS].mean().round(3))

## 5. Preprocesamiento BASE para BETO y DistilBETO

Esta versión realiza una limpieza mínima:

- normalización Unicode NFKC y eliminación de caracteres de control/formato (esto incluye el *zero-width joiner*, por lo que los emojis compuestos quedan separados en sus componentes);
- saltos de línea y espacios múltiples → un espacio;
- URL → `url`;
- mención → `@usuario`;
- elimina los marcadores aislados `RT` / `FAV` (sin distinguir mayúsculas).

Se conservan:

- mayúsculas;
- tildes;
- puntuación;
- `¿?` / `¡!`;
- hashtags;
- emojis;
- elongaciones;
- stopwords.

**DistilBETO es uncased:** no hace falta convertir manualmente el texto a minúsculas; se deja esa responsabilidad a su tokenizer oficial.

In [ ]:
URL_RE = re.compile(r"(?:https?://\S+|www\.\S+)", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"(?<!\w)@\w+")
RT_FAV_RE = re.compile(r"\b(?:RT|FAV)\b", flags=re.IGNORECASE)


def normalizar_unicode_y_espacios(texto):
    """Normaliza Unicode (NFKC), elimina caracteres de control/formato y colapsa espacios."""
    texto = unicodedata.normalize("NFKC", str(texto))
    texto = "".join(
        c for c in texto
        if unicodedata.category(c)[0] != "C" or c in "\n\t"
    )
    return re.sub(r"\s+", " ", texto).strip()


def preprocesar_transformer_base(texto):
    """Limpieza mínima: Unicode, URLs, menciones y marcadores RT/FAV."""
    texto = normalizar_unicode_y_espacios(texto)
    texto = URL_RE.sub(" url ", texto)
    texto = MENTION_RE.sub(" @usuario ", texto)
    texto = RT_FAV_RE.sub(" ", texto)
    return re.sub(r"\s+", " ", texto).strip()

## 6. Preprocesamiento SOCIAL para BETO y DistilBETO

La literatura sobre Twitter en español deja abiertas algunas decisiones para BETO/DistilBETO.

Por eso esta segunda variante, que parte del texto BASE, permite comprobar posteriormente, mediante validación, si ayuda:

- limitar a 3 cualquier carácter repetido 4 o más veces;
- transformar hashtags en contenido textual (`#NoMasCorrupcion` → `hashtag No Mas Corrupcion`);
- convertir emojis a descripción textual en español, delimitada como `emoji … emoji`.

La puntuación y las stopwords se siguen conservando.

In [ ]:
def limitar_elongaciones(texto, max_repeticiones=3):
    """Reduce a max_repeticiones cualquier carácter repetido más veces."""
    patron = re.compile(r"(.)\1{" + str(max_repeticiones) + r",}", flags=re.IGNORECASE)
    return patron.sub(lambda m: m.group(1) * max_repeticiones, texto)


def separar_camel_case(token):
    """Separa palabras unidas en un hashtag (camelCase, guiones bajos y números)."""
    token = token.replace("_", " ")
    token = re.sub(r"(?<=[a-záéíóúüñ])(?=[A-ZÁÉÍÓÚÜÑ])", " ", token)
    token = re.sub(r"(?<=[A-Za-zÁÉÍÓÚÜÑáéíóúüñ])(?=\d)", " ", token)
    token = re.sub(r"(?<=\d)(?=[A-Za-zÁÉÍÓÚÜÑáéíóúüñ])", " ", token)
    return re.sub(r"\s+", " ", token).strip()


def normalizar_hashtags(texto):
    """Convierte #Hashtag en 'hashtag' + su contenido separado en palabras."""
    def reemplazar(match):
        contenido = separar_camel_case(match.group(1))
        return f" hashtag {contenido} "

    return re.sub(r"#([\wÁÉÍÓÚÜÑáéíóúüñ]+)", reemplazar, texto)


def emojis_a_texto_es(texto):
    """Convierte cada emoji en su descripción en español, delimitada por 'emoji'.
    Ejemplo aproximado: 🙄 -> emoji cara con ojos en blanco emoji"""
    texto = emoji.demojize(texto, language="es")
    patron = re.compile(r":([^:\s]+):")

    def reemplazar(match):
        descripcion = match.group(1).replace("_", " ")
        return f" emoji {descripcion} emoji "

    return patron.sub(reemplazar, texto)


def preprocesar_transformer_social(texto):
    """BASE + elongaciones limitadas + hashtags y emojis como texto."""
    texto = preprocesar_transformer_base(texto)
    texto = limitar_elongaciones(texto, max_repeticiones=3)
    texto = normalizar_hashtags(texto)
    texto = emojis_a_texto_es(texto)
    return re.sub(r"\s+", " ", texto).strip()

## 7. Preprocesamiento para RoBERTuito

RoBERTuito fue entrenado específicamente para redes sociales en español.

Se utiliza directamente su preprocesamiento oficial, sobre el texto original:

```python
pysentimiento.preprocessing.preprocess_tweet(texto, lang="es", shorten=3)
```

en lugar de replicar manualmente su lógica. Si `pysentimiento` no está instalado, la columna `MESSAGE_ROBERTUITO` se exporta vacía.

In [ ]:
def preprocesar_robertuito(texto):
    """Preprocesamiento oficial de RoBERTuito (pysentimiento)."""
    if not PYSENTIMIENTO_DISPONIBLE:
        raise ImportError(
            "Falta pysentimiento. Instala con `pip install pysentimiento` o `uv add pysentimiento`."
        )
    return preprocess_tweet(str(texto), lang="es", shorten=3)

## 8. Generación de textos para Transformers

Se aplican las tres representaciones al train y al test, con las mismas reglas.

In [ ]:
for nombre, df in [("Train", df_train), ("Test", df_test)]:
    print(f"Procesando {nombre}...")
    df["MESSAGE_TRANSFORMER_BASE"] = (
        df["MESSAGE"].fillna("").astype(str).apply(preprocesar_transformer_base)
    )
    df["MESSAGE_TRANSFORMER_SOCIAL"] = (
        df["MESSAGE"].fillna("").astype(str).apply(preprocesar_transformer_social)
    )
    if PYSENTIMIENTO_DISPONIBLE:
        df["MESSAGE_ROBERTUITO"] = (
            df["MESSAGE"].fillna("").astype(str).apply(preprocesar_robertuito)
        )
    else:
        df["MESSAGE_ROBERTUITO"] = pd.NA
        print("  AVISO: RoBERTuito no se procesó porque falta pysentimiento.")

print("\nEjemplos por variante:")
for variante in VARIANTES:
    row = df_train[df_train["VARIANTE"] == variante].iloc[0]
    print(f"\n--- {variante.upper()} ---")
    print("ORIGINAL:  ", row["MESSAGE"])
    print("BASE:      ", row["MESSAGE_TRANSFORMER_BASE"])
    print("SOCIAL:    ", row["MESSAGE_TRANSFORMER_SOCIAL"])
    if PYSENTIMIENTO_DISPONIBLE:
        print("ROBERTUITO:", row["MESSAGE_ROBERTUITO"])

## 9. Diagnóstico de tokenización — solo train

Para cada tokenizer (BETO, DistilBETO y, si `pysentimiento` está disponible, RoBERTuito), variante dialectal y representación se calculan estadísticas descriptivas de longitud: percentiles 50/95/99, máximo, % de textos con más de 64 y 128 tokens, tokens por palabra y `[UNK]` por cada 1000 tokens.

Este diagnóstico utiliza únicamente el train y no modifica registros.

In [ ]:
def cargar_tokenizers():
    """Carga los tokenizers oficiales de cada checkpoint."""
    if not TRANSFORMERS_DISPONIBLE:
        raise ImportError(
            "Falta transformers. Instala con `pip install transformers` o `uv add transformers`."
        )
    tokenizers = {
        "BETO": AutoTokenizer.from_pretrained(CHECKPOINTS["BETO"], use_fast=True),
        "DistilBETO": AutoTokenizer.from_pretrained(CHECKPOINTS["DistilBETO"], use_fast=True),
    }
    if PYSENTIMIENTO_DISPONIBLE:
        tokenizers["RoBERTuito"] = AutoTokenizer.from_pretrained(
            CHECKPOINTS["RoBERTuito"], use_fast=True
        )
    return tokenizers


def calcular_stats_tokenizacion(tokenizer, textos):
    """Estadísticas de longitud en tokens para una lista de textos."""
    longitudes = []
    unk = 0
    total_tokens = 0
    total_palabras = 0
    unk_id = tokenizer.unk_token_id

    for texto in textos:
        texto = str(texto)
        ids = tokenizer(
            texto, add_special_tokens=True, truncation=False, padding=False
        )["input_ids"]

        longitudes.append(len(ids))
        total_tokens += len(ids)
        total_palabras += max(len(texto.split()), 1)
        if unk_id is not None:
            unk += sum(token == unk_id for token in ids)

    arr = np.asarray(longitudes, dtype=np.int32)
    return {
        "n": len(arr),
        "p50": np.percentile(arr, 50),
        "p95": np.percentile(arr, 95),
        "p99": np.percentile(arr, 99),
        "max": int(arr.max()),
        "pct_gt_64": 100 * (arr > 64).mean(),
        "pct_gt_128": 100 * (arr > 128).mean(),
        "tokens_por_palabra": total_tokens / max(total_palabras, 1),
        "unk_por_1000_tokens": 1000 * unk / max(total_tokens, 1),
    }


def diagnosticar_train():
    """Aplica el diagnóstico a cada combinación tokenizer × representación × variante."""
    tokenizers = cargar_tokenizers()
    configs = [
        ("BETO_BASE", "BETO", "MESSAGE_TRANSFORMER_BASE"),
        ("BETO_SOCIAL", "BETO", "MESSAGE_TRANSFORMER_SOCIAL"),
        ("DISTILBETO_BASE", "DistilBETO", "MESSAGE_TRANSFORMER_BASE"),
        ("DISTILBETO_SOCIAL", "DistilBETO", "MESSAGE_TRANSFORMER_SOCIAL"),
    ]
    if PYSENTIMIENTO_DISPONIBLE:
        configs.append(("ROBERTUITO", "RoBERTuito", "MESSAGE_ROBERTUITO"))

    filas = []
    for cfg, tokenizer_name, columna in configs:
        tok = tokenizers[tokenizer_name]
        for variante in VARIANTES:
            textos = (
                df_train.loc[df_train["VARIANTE"] == variante, columna]
                .fillna("").astype(str).tolist()
            )
            filas.append({
                "configuracion": cfg,
                "tokenizer": tokenizer_name,
                "variante": variante,
                **calcular_stats_tokenizacion(tok, textos),
            })
    return pd.DataFrame(filas)


if TRANSFORMERS_DISPONIBLE:
    df_tokenizacion = diagnosticar_train()
    display(
        df_tokenizacion.style.format({
            "p50": "{:.1f}",
            "p95": "{:.1f}",
            "p99": "{:.1f}",
            "pct_gt_64": "{:.2f}%",
            "pct_gt_128": "{:.2f}%",
            "tokens_por_palabra": "{:.3f}",
            "unk_por_1000_tokens": "{:.3f}",
        })
    )
else:
    df_tokenizacion = None
    print("Instala transformers para ejecutar el diagnóstico.")

## 10. Exportación final por variante

Se exportan 6 archivos:

- 3 variantes dialectales (`mx`, `es`, `cu`);
- 2 splits (`train`, `test`).

Cada archivo contiene las tres columnas de texto (`BASE`, `SOCIAL`, `ROBERTUITO`) y las 9 variables lingüísticas; es la entrada común de BETO, DistilBETO y RoBERTuito. También se guardan el diagnóstico de tokenización, la auditoría del train, las filas retiradas por conflicto y un manifest con el número de filas por variante.

El preprocesamiento no elimina registros del train en función del contenido del test.

In [ ]:
COLUMNAS_BASE = ["ID", "TOPIC", "IS_IRONIC", "MESSAGE", "VARIANTE"] + FEATURE_COLS

COLUMNAS_TEXTO = [
    "MESSAGE_TRANSFORMER_BASE",
    "MESSAGE_TRANSFORMER_SOCIAL",
    "MESSAGE_ROBERTUITO",
]


def exportar_por_variante(df, split):
    """Exporta un CSV por variante con las columnas base y los tres textos Transformer."""
    for variante in VARIANTES:
        salida = (
            df.loc[df["VARIANTE"] == variante, COLUMNAS_BASE + COLUMNAS_TEXTO]
            .reset_index(drop=True)
        )
        ruta = OUT_DIR / f"{split}_transformers_{variante}.csv"
        salida.to_csv(ruta, index=False)
        print(f"[EXPORT][{split.upper()}][{variante.upper()}] {len(salida)} registros -> {ruta}")


print("=" * 78)
print("EXPORTACIÓN FINAL — TRANSFORMERS")
print("=" * 78)

for split, df in [("train", df_train), ("test", df_test)]:
    exportar_por_variante(df, split)

if df_tokenizacion is not None:
    ruta_diag = OUT_DIR / "diagnostico_tokenizacion_train.csv"
    df_tokenizacion.to_csv(ruta_diag, index=False)
    print(f"\n[EXPORT][AUDIT] Diagnóstico de tokenización -> {ruta_diag}")


# Auditoría interna del train.
df_auditoria_train.to_csv(OUT_DIR / "auditoria_duplicados_train.csv", index=False)

if len(df_conflictos_train) > 0:
    df_conflictos_train.to_csv(OUT_DIR / "conflictos_etiqueta_train.csv", index=False)


# Manifest mínimo: permite comprobar versión y cantidad de filas.
manifest = pd.DataFrame([
    {
        "pipeline_version": "transformers_comun_v1",
        "variante": variante,
        "train_final": int((df_train["VARIANTE"] == variante).sum()),
        "test_n": int((df_test["VARIANTE"] == variante).sum()),
    }
    for variante in VARIANTES
])
manifest.to_csv(OUT_DIR / "manifest_preprocesamiento_transformers.csv", index=False)

print("\nMANIFEST DEL PREPROCESAMIENTO")
display(manifest)

print("\n[PREPROCESS][Transformers] EXPORTACIÓN: OK")

## 11. Uso posterior

BETO, DistilBETO y RoBERTuito cargan directamente:

```text
data/transformers/train_transformers_{mx,es,cu}.csv
data/transformers/test_transformers_{mx,es,cu}.csv
```

Según el diseño previsto, cada modelo construye después sus propios grupos de textos coincidentes a partir de sus representaciones.

El preprocesamiento no decide los folds ni elimina registros en función del test.

## 12. Resumen metodológico

### Train

- control interno de duplicados exactos y conflictos de etiqueta;
- extracción de las 9 variables lingüísticas;
- generación de las representaciones `BASE`, `SOCIAL` y, si está disponible, `ROBERTUITO`;
- diagnóstico de tokenización;
- exportación.

### Modelos

Según el diseño previsto, BETO, DistilBETO y RoBERTuito:

1. cargan el train preprocesado;
2. construyen sus propios grupos de textos coincidentes;
3. ejecutan `StratifiedGroupKFold` 5×3;
4. seleccionan hiperparámetros con F1-Macro.

### Test

Se mantiene como conjunto de evaluación final y no determina qué registros permanecen en train.